# 🚜 Pipeline d'Entraînement et d'Évaluation des Modèles ML

Ce notebook exécute le script d'entraînement `train.py` et visualise les performances du meilleur modèle de Machine Learning obtenu (Random Forest ou XGBoost).

In [ ]:
# 1. Lancer l'entraînement complet des modèles
%run train.py

[INFO] Chargement et fusion des donnees...
[SUCCESS] Donnees fusionnees avec succes. Nombre total de lignes : 819,408
[INFO] Feature Engineering...

[INFO] Division temporelle des donnees...
   Train set : 658,603 lignes (annees <= 2013)
   Test set  : 160,805 lignes (annees > 2013)

[INFO] Entrainement de Baseline Ridge...


c:\Users\alexi\Desktop\Swarm_Prediction\.venv\Lib\site-packages\sklearn\impute\_base.py:647: UserWarning: Skipping features without any observed values: ['Bilan_sols_kgha']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(



===== Evaluation du modele : Baseline Ridge =====


c:\Users\alexi\Desktop\Swarm_Prediction\.venv\Lib\site-packages\sklearn\impute\_base.py:647: UserWarning: Skipping features without any observed values: ['Bilan_sols_kgha']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
c:\Users\alexi\Desktop\Swarm_Prediction\.venv\Lib\site-packages\sklearn\impute\_base.py:647: UserWarning: Skipping features without any observed values: ['Bilan_sols_kgha']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(


Train R2 : 0.6666 | Test R2 : 0.6032
Train MAE: 3211.22 kg/ha | Test MAE: 4452.01 kg/ha
Train RMSE: 6806.54 kg/ha | Test RMSE: 8830.62 kg/ha

[INFO] Entrainement de Random Forest...


c:\Users\alexi\Desktop\Swarm_Prediction\.venv\Lib\site-packages\sklearn\impute\_base.py:647: UserWarning: Skipping features without any observed values: ['Bilan_sols_kgha']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(


## 📊 Importance des Features

Visualisons les variables qui influencent le plus les prédictions du rendement de notre modèle.

In [ ]:
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Charger le meilleur modèle
pipeline = joblib.load("models/best_model_pipeline.joblib")
model = pipeline.named_steps["model"]
preprocessor = pipeline.named_steps["preprocessor"]

# Récupérer les noms des features issues du preprocessor
feature_names = preprocessor.get_feature_names_out()
importances = model.feature_importances_

# DataFrame
df_imp = pd.DataFrame({"Feature": feature_names, "Importance": importances})
# Rendre les noms des features plus lisibles
df_imp["Feature"] = df_imp["Feature"].str.replace("num__", "").str.replace("cat__", "")
df_imp = df_imp.sort_values(by="Importance", ascending=False).head(15)

# Tracer
plt.figure(figsize=(12, 6))
sns.barplot(data=df_imp, x="Importance", y="Feature", palette="viridis")
plt.title("Top 15 des Features les Plus Importantes dans le Modèle", fontsize=14, fontweight="bold")
plt.xlabel("Importance relative")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()

## 📈 Prédictions vs Valeurs Réelles

Comparons les rendements prédits par le modèle avec les rendements réellement observés sur le jeu de test (données après 2013).

In [ ]:
# Charger l'échantillon de test généré par train.py
df_test = pd.read_csv("data/cleaned/test_sample.csv")

num_cols = [
    "Annee", "Temperature_C", "Temperature_C_sq", 
    "Precipitations_mm", "Precipitations_mm_sq", 
    "Engrais_kgha", "Pesticides_kgha", "Bilan_sols_kgha",
    "Engrais_Temp_interaction"
]
cat_cols = ["Pays", "Produit"]

X_test_raw = df_test[num_cols + cat_cols]
y_test_real = df_test["Rendement_kgha"]

# Prédictions du pipeline et passage à l'échelle d'origine
y_pred_log = pipeline.predict(X_test_raw)
y_pred_real = np.expm1(y_pred_log)

# Graphique
plt.figure(figsize=(10, 8))
plt.scatter(y_test_real, y_pred_real, alpha=0.4, color="teal", edgecolors="grey")
plt.plot([0, 100000], [0, 100000], color="red", lw=2, ls="--", label="Prédiction Parfaite")
plt.title("Prédictions vs Valeurs Réelles (Rendement en kg/ha)", fontsize=14, fontweight="bold")
plt.xlabel("Rendement Réel (kg/ha)")
plt.ylabel("Rendement Prédit (kg/ha)")
plt.xlim(0, 100000)
plt.ylim(0, 100000)
plt.legend()
plt.tight_layout()
plt.show()

## 🔍 Analyse des Résidus (Erreurs de Prédiction)

Analysons si les erreurs sont bien réparties ou si le modèle fait des erreurs systématiques.

In [ ]:
residuals = y_test_real - y_pred_real

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Distribution des résidus
sns.histplot(residuals, bins=50, kde=True, ax=axes[0], color="tomato")
axes[0].set_title("Distribution des Résidus de Prédiction", fontsize=12)
axes[0].set_xlabel("Erreur de prédiction (kg/ha)")
axes[0].set_ylabel("Nombre de mesures")

# Résidus vs Prédictions
axes[1].scatter(y_pred_real, residuals, alpha=0.3, color="steelblue")
axes[1].axhline(0, color="red", lw=2, ls="--")
axes[1].set_title("Résidus vs Valeurs Prédites", fontsize=12)
axes[1].set_xlabel("Rendement Prédit (kg/ha)")
axes[1].set_ylabel("Erreur (kg/ha)")
axes[1].set_xlim(0, 100000)

plt.tight_layout()
plt.show()